In [1]:
import os
import matplotlib as mpl
import scanpy as sc
import pandas as pd

from pathlib import Path
from hgctsvg.preprocessing import common, image, kidneycancer
from pybiomart import Dataset

/opt/conda/envs/HGCTSVG/lib/python3.12/site-packages/numcodecs/__init__.py:106: DeprecationWarning: crc32c usage is deprecated since numcodecs v0.16.4. It is recommended to install google_crc32c instead.
  from numcodecs.checksum32 import CRC32, Adler32, JenkinsLookup3
/opt/conda/envs/HGCTSVG/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/opt/conda/envs/HGCTSVG/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFou

In [2]:
data_dir = Path('data_repo/Human Kidney Cancer')
patient_id = 'PD47171'

Specify the output folder.

In [3]:
spotiphy_proj_folder = Path("spotiphy_project_files/Human Kidney Cancer Interface/")
spotiphy_proj_folder.mkdir(parents=True, exist_ok=True)    

## Data Preparation: Loading and Processing <a id=Load-data></a>

Load the scRNA data and visium data.

In [4]:
for filename in data_dir.iterdir():
    print(filename)

data_repo/Human Kidney Cancer/visium_merge_core_upload.h5ad
data_repo/Human Kidney Cancer/visium_merge_inter_upload.h5ad
data_repo/Human Kidney Cancer/RCC_upload_final_raw_counts.h5ad
data_repo/Human Kidney Cancer/.ipynb_checkpoints


In [5]:
adata_sc = sc.read_h5ad(data_dir / "RCC_upload_final_raw_counts.h5ad")
adata_st_inter = sc.read_h5ad(data_dir / "visium_merge_inter_upload.h5ad")

In [8]:
# Extract data of patient PD47171
adata_sc = kidneycancer.extract_sc_patient_data(adata_sc, patient_id)
adata_st_inter = kidneycancer.extract_st_patient_data(adata_st_inter, patient_id)

In [9]:
adata_sc, adata_st_inter

(AnnData object with n_obs × n_vars = 36110 × 19736
     obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'patient', 'percent.mt', 'summaryDescription', 'annotation', 'region', 'broad_type'
     var: 'name'
     uns: 'annotation_colors', 'broad_type_colors', 'region_colors', 'summaryDescription_colors'
     obsm: 'X_pca', 'X_umap',
 AnnData object with n_obs × n_vars = 2048 × 36601
     obs: 'in_tissue', 'array_row', 'array_col', 'sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'mt_frac', 'donor_id', 'region'
     var: 'feature_types', 'genome', 'SYMBOL', 'mt', 'n_cells_by_counts-6800STDY12499411', 'mean_counts-6800STDY12499411', 'log1p_mean_counts-6800STDY12499411', 'pct_dropout_by_counts-6800STDY12499411', 'total_counts-6800STDY12499411', 'log1p_total_counts-6800STDY12499411'
     uns: 'sample_colors', 'spatial'
    

In [10]:
geneset = common.get_annotation(host="feb2021.archive.ensembl.org")

HTTPError: 500 Server Error: Internal Server Error for url: http://feb2021.archive.ensembl.org:80/biomart/martservice?type=datasets&mart=ENSEMBL_MART_ENSEMBL

In [ ]:
conflicts = common.find_conflicting_genes(adata_sc, adata_st_inter, geneset)
conflicts

In [ ]:
drop_cols = [
    'ENSG00000286070',  # GGT1
    'ENSG00000280987',  # MATR3
    'ENSG00000285053',  # TBCE
]

rename_cols = {
    'ENSG00000284024': 'MSANTD7',
    'ENSG00000269226': 'TMSB15C'
}

In [ ]:
# TODO
# TMSB15B -> rename ENSG00000269226 to TMSB15C
# MATR3 -> drop ENS00000280987
# TBCE -> drop ENSG00000285053
new_adata_sc, new_adata_st_inter = common.use_ensembl_gene_ids(adata_sc, 
                                                         adata_st_inter,
                                                         geneset,
                                                         drop_cols = drop_cols, 
                                                         rename_cols = rename_cols)

In [ ]:
new_adata_sc, new_adata_st_inter